In [1]:
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# logging.basicConfig(level=logging.DEBUG)
n_epoch = 1000
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescent(model.parameters())

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.36112165451049805
epoch:  1, loss: 0.359561562538147
epoch:  2, loss: 0.3580087125301361
epoch:  3, loss: 0.35646316409111023
epoch:  4, loss: 0.3549247980117798
epoch:  5, loss: 0.3533936142921448
epoch:  6, loss: 0.35186949372291565
epoch:  7, loss: 0.3503524363040924
epoch:  8, loss: 0.3488422930240631
epoch:  9, loss: 0.3473392426967621
epoch:  10, loss: 0.34584325551986694
epoch:  11, loss: 0.3443542420864105
epoch:  12, loss: 0.3428722321987152
epoch:  13, loss: 0.34139707684516907
epoch:  14, loss: 0.3399289548397064
epoch:  15, loss: 0.33846765756607056
epoch:  16, loss: 0.3370131254196167
epoch:  17, loss: 0.33556532859802246
epoch:  18, loss: 0.33412426710128784
epoch:  19, loss: 0.3326897621154785
epoch:  20, loss: 0.3312619626522064
epoch:  21, loss: 0.32984092831611633
epoch:  22, loss: 0.3284265100955963
epoch:  23, loss: 0.32701852917671204
epoch:  24, loss: 0.3256171643733978
epoch:  25, loss: 0.3242223560810089
epoch:  26, loss: 0.3228340744972229
ep

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -5424.39111328125
Test metrics:  R2 = -7449.49365234375


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLipschitz(model.parameters())

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.5242754220962524
epoch:  1, loss: 0.045854225754737854
epoch:  2, loss: 0.03564050421118736
epoch:  3, loss: 0.03556200489401817
epoch:  4, loss: 0.03554760664701462
epoch:  5, loss: 0.03546028956770897
epoch:  6, loss: 0.03484981134533882
epoch:  7, loss: 0.03448816388845444
epoch:  8, loss: 0.038748227059841156
epoch:  9, loss: 0.034293368458747864
epoch:  10, loss: 0.034108541905879974
epoch:  11, loss: 0.034082237631082535
epoch:  12, loss: 0.03363003954291344
epoch:  13, loss: 0.028939303010702133
epoch:  14, loss: 0.046002600342035294
epoch:  15, loss: 0.3290863037109375
epoch:  16, loss: 0.02776927500963211
epoch:  17, loss: 0.027282828465104103
epoch:  18, loss: 0.025946706533432007
epoch:  19, loss: 0.02274511754512787
epoch:  20, loss: 0.02463943511247635
epoch:  21, loss: 0.022253699600696564
epoch:  22, loss: 0.016915610060095787
epoch:  23, loss: 0.01643955335021019
epoch:  24, loss: 0.013892024755477905
epoch:  25, loss: 0.10119487345218658
epoch:  26, 

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9271361827850342
Test metrics:  R2 = 0.9234766364097595


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model.parameters())

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.31840312480926514
epoch:  1, loss: 0.17044690251350403
epoch:  2, loss: 0.09877011179924011
epoch:  3, loss: 0.06407377868890762
epoch:  4, loss: 0.047763340175151825
epoch:  5, loss: 0.040449295192956924
epoch:  6, loss: 0.037292495369911194
epoch:  7, loss: 0.03596707805991173
epoch:  8, loss: 0.035419948399066925
epoch:  9, loss: 0.035195592790842056
epoch:  10, loss: 0.03510291129350662
epoch:  11, loss: 0.035063352435827255
epoch:  12, loss: 0.035045016556978226
epoch:  13, loss: 0.035035181790590286
epoch:  14, loss: 0.03501560166478157
epoch:  15, loss: 0.034994736313819885
epoch:  16, loss: 0.03498386591672897
epoch:  17, loss: 0.03496505320072174
epoch:  18, loss: 0.0349416546523571
epoch:  19, loss: 0.0349295400083065
epoch:  20, loss: 0.03491158038377762
epoch:  21, loss: 0.034884486347436905
epoch:  22, loss: 0.034870728850364685
epoch:  23, loss: 0.03485195338726044
epoch:  24, loss: 0.034820836037397385
epoch:  25, loss: 0.034804973751306534
epoch:  26,

In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7045339345932007
Test metrics:  R2 = 0.7130573987960815


In [18]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentTR(model.parameters(), radius_init=1e-3)

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.6166330575942993
epoch:  1, loss: 0.6166330575942993
epoch:  2, loss: 0.6166330575942993
epoch:  3, loss: 0.6166330575942993
epoch:  4, loss: 0.6166330575942993
epoch:  5, loss: 0.6166330575942993
epoch:  6, loss: 0.6166330575942993
epoch:  7, loss: 0.6166330575942993
epoch:  8, loss: 0.6166330575942993
epoch:  9, loss: 0.6166330575942993
epoch:  10, loss: 0.6166330575942993
epoch:  11, loss: 0.6166330575942993
epoch:  12, loss: 0.6166330575942993
epoch:  13, loss: 0.6166330575942993
epoch:  14, loss: 0.6166330575942993
epoch:  15, loss: 0.6166330575942993
epoch:  16, loss: 0.6166330575942993
epoch:  17, loss: 0.6166330575942993
epoch:  18, loss: 0.6166330575942993
epoch:  19, loss: 0.6166330575942993
epoch:  20, loss: 0.6166330575942993
epoch:  21, loss: 0.6166330575942993
epoch:  22, loss: 0.6166330575942993
epoch:  23, loss: 0.6166330575942993
epoch:  24, loss: 0.6166330575942993
epoch:  25, loss: 0.6166330575942993
epoch:  26, loss: 0.6166330575942993
epoch:  27,

In [12]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -2583.70556640625
Test metrics:  R2 = -3437.39892578125
